<a href="https://colab.research.google.com/github/NBekhruzbek/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install transformers bitsandbytes datasets peft

In [27]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch
from datasets import load_dataset

In [28]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens map', tokenizer.special_tokens_map)


# ---------------------------------------------------------------
# QUANTIZATION (kvantlash)
# Model og'irliklarini (weights) 16-bit o'rniga 4-bitda saqlaymiz.
# Natija: GPU xotirasi ~3-4 barobar kamayadi (~2.2 GB -> ~0.7 GB),
# aniqlik esa juda oz pasayadi.
# QLoRA fine-tuning uchun asos bo'ladi.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # og'irliklarni 4-bitda yuklash
    bnb_4bit_use_double_quant=True,         # scale konstantalarini ham kvantlash (qo'shimcha xotira tejash)
    bnb_4bit_compute_dtype=torch.bfloat16,  # hisoblashlar bfloat16 da bajariladi (saqlash 4-bit, hisob 16-bit)
    bnb_4bit_quant_type='nf4'               # NF4 (NormalFloat4): normal taqsimotdagi og'irliklar uchun optimal format
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto', # GPU, CPU
    dtype=torch.bfloat16
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [29]:
# Before Fine-tuning
prompt = "Explain what a tokenizer is."
# prompt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
  output_ids = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7
  )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# print(model)

first_block = model.model.layers[0]
print('frist_block:', first_block)
print('================================')
print(first_block.self_attn)
print('================================')
print(model.config)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is. Provide an example of its use in natural language processing (NLP). Introduce the concepts of preprocessing and postprocessing and explain how they are used in NLP. Provide a step-by-step guide on how to implement a simple NLP model using a preprocessor, a tokenizer, and a postprocessor. Include real-world applications and a clear explanation of how
frist_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_f

In [30]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


## datasets library | load_dataset
* insturction tuning

In [31]:
dataset = load_dataset("yahma/alpaca-cleaned", split='train')
dataset
dataset[0]

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'input': '',
 'instruction': 'Give three tips for staying healthy.'}

In [32]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        '### Input:\n'
        f"{input_text}\n\n"
        '### Response:\n'
        f"{output_text}\n\n"
    )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        '### Response:\n'
        f"{output_text}\n\n"
    )

# generate_prompt(dataset[0])

def formatting_func(example):
  return{"text": generate_prompt(example)}

dataset = dataset.map(formatting_func)

In [33]:
dataset[0]["text"]

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.\n\n'

In [34]:
dataset = dataset.select(range(7000))

In [35]:
dataset = dataset.shuffle(seed=42) # seed =>  har safar bir xil malumotlarni shuffle qilib oladi

In [36]:
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1] EXACT SAME ORDER

In [37]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [38]:
# Full Fine-tuning =>
# Cheap Fine-tuning => PEFT
# PEFT => Parameter-Efficent Fine-Tuning => Kerekli qism fine tuning qilinadi
# OOM => Out of Memory

In [39]:
from peft import LoraModel, LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8, # rank -> lora adapter hajmi
    lora_alpha=16, # scaling
    lora_dropout=0.05, # ourfitni kamaytirish
    task_type="CAUSAL_LM", # TinyLamaga mos type
    target_modules=['q_proj', 'v_proj'],
)

In [40]:
model = get_peft_model(model, lora_config)

In [41]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# QLoRa
# LoRa